Importation des bibliothèques

In [3]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from config import SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET, SPOTIPY_REDIRECT_URI
import requests
import pandas as pd
from IPython.display import display
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

Initialiser le client Spotify

In [4]:
# 1. Obtenir un token (Client Credentials Flow)
def get_spotify_token():
    auth_url = "https://accounts.spotify.com/api/token"
    auth_response = requests.post(
        auth_url,
        data={"grant_type": "client_credentials"},
        auth=(SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET)
    )
    auth_response.raise_for_status()
    return auth_response.json()["access_token"]
# Portées (scopes) nécessaires pour récupérer tes morceaux


SCOPE = [
    "user-read-recently-played",  # Pour récupérer ton historique récent
    "user-top-read"                # Pour récupérer tes morceaux préférés
]

# Initialise le client Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET,
    redirect_uri=SPOTIPY_REDIRECT_URI,
    scope=SCOPE
))

In [12]:
# 1. Obtenir un token (Client Credentials Flow)
def get_spotify_token():
    auth_url = "https://accounts.spotify.com/api/token"
    auth_response = requests.post(
        auth_url,
        data={"grant_type": "client_credentials"},
        auth=(SPOTIPY_CLIENT_ID, SPOTIPY_CLIENT_SECRET)
    )
    auth_response.raise_for_status()
    return auth_response.json()["access_token"]
# Portées (scopes) nécessaires pour récupérer tes morceaux


SCOPE = [
    "user-read-recently-played",  # Pour récupérer ton historique récent
    "user-top-read"                # Pour récupérer tes morceaux préférés
]

# Initialise le client Spotify
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=SPOTIPY_CLIENT_ID,
    client_secret=SPOTIPY_CLIENT_SECRET,
    redirect_uri=SPOTIPY_REDIRECT_URI,
    scope=SCOPE
))

récupérer les 10 dernièes chansons écoutées avec l'API web de Spotify

In [7]:
# Récupère tes 10 derniers morceaux écoutés
recent_tracks = sp.current_user_recently_played(limit=10)

print("🎵 Tes 10 derniers morceaux écoutés :")
for idx, item in enumerate(recent_tracks['items'], 1):
    track = item['track']
    artists = ', '.join([artist['name'] for artist in track['artists']])
    played_at = item['played_at']  # Date et heure d'écoute
    print(f"{idx}. {track['name']} - {artists} (ID: {track['id']}) | Écoute le {played_at}")

🎵 Tes 10 derniers morceaux écoutés :
1. Vendredi 13 Mars - Bigflo & Oli (ID: 1BAFwMW6iE4Wp4PJR4KEJy) | Écoute le 2026-09-29T15:39:57.557Z
2. Taki Taki (feat. Selena Gomez, Ozuna & Cardi B) - DJ Snake, Selena Gomez, Ozuna, Cardi B (ID: 0KoiMHhqKLoL4xM54rcWYY) | Écoute le 2026-09-29T15:35:33.316Z
3. Amnésia - Jul (ID: 6wo88ISzGpdCSPl0yTibhJ) | Écoute le 2026-09-29T15:32:00.445Z
4. Ma mélodie (feat. GIMS) - Dry, GIMS (ID: 0BU5e8rDml803bp7mbouC0) | Écoute le 2026-09-29T15:26:17.776Z
5. Sale journée (feat. Bigflo & Oli) - Black M, Bigflo & Oli (ID: 5f37dB1I9qau8H0llB7ziG) | Écoute le 2026-09-29T15:19:48.199Z
6. Gunshot - GIMS (ID: 1EZ7C9q6j11X0UxgXGTBrN) | Écoute le 2026-09-29T15:15:33.386Z
7. La classe - Jul (ID: 4T5cEAqAvRee5VAIHSex82) | Écoute le 2026-09-29T15:10:43.715Z
8. La terre est ronde - Orelsan (ID: 2wyreldlNuoBG7ju3fUynK) | Écoute le 2026-09-29T15:07:03.147Z
9. Warano-Style - GIMS (ID: 7JbaSH5ZvdmKHxsjPXfPCk) | Écoute le 2026-09-29T15:02:50.300Z
10. And We Danced (feat. Ziggy St

Aller chercher les audio features de toutes les chansons grâce à l'API Reccobeats

In [ ]:
# API reccobeats to have the audio features
RECCO_URL = "https://api.reccobeats.com/v1/audio-features"
#function to get audio features
def get_audio_features(spotify_ids):
    """Récupère les audio features via ReccoBeats à partir d'IDs Spotify."""
    response = requests.get(
        RECCO_URL,
        params={"ids": ",".join(spotify_ids)},
        headers={"Accept": "application/json"},
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    # La réponse est généralement de la forme {"content": [ ... ]}
    return data.get("content", data) if isinstance(data, dict) else data


track_ids = [item["track"]["id"] for item in recent_tracks["items"]]
track_names = {item["track"]["id"]: item["track"]["name"] for item in recent_tracks["items"]}

# get audio features of track_ids
features = get_audio_features(track_ids)

Créer une matrice contenant les audio features de toutes les chansons

In [10]:
#create a dataframe of features
features_df = pd.DataFrame(features)
matrix = features_df.iloc[1:11, 3:14]

feature_columns = ["acousticness", "danceability", "energy", "instrumentalness",
                    "key", "liveness", "loudness", "mode", "speechiness", "tempo", "valence"]
features_matrix = matrix[feature_columns].values  # Matrice numpy des features

print("Matrice des features :")
print(features_matrix)

Matrice des features :
[[ 1.58000e-01  8.43000e-01  8.02000e-01  5.00000e-06  1.00000e+00
   6.42000e-02 -4.16700e+00  0.00000e+00  2.28000e-01  9.58810e+01
   6.18000e-01]
 [ 2.76000e-01  7.98000e-01  7.38000e-01  0.00000e+00  9.00000e+00
   6.62000e-02 -5.13900e+00  0.00000e+00  3.41000e-01  9.73460e+01
   7.81000e-01]
 [ 4.07000e-02  8.23000e-01  9.66000e-01  1.85000e-01  5.00000e+00
   2.56000e-01 -4.62000e+00  0.00000e+00  9.80000e-02  1.28033e+02
   5.90000e-01]
 [ 3.01000e-03  7.41000e-01  7.95000e-01  0.00000e+00  0.00000e+00
   9.08000e-02 -5.31500e+00  1.00000e+00  1.09000e-01  9.20140e+01
   3.34000e-01]
 [ 9.90000e-02  6.34000e-01  6.75000e-01  3.12000e-04  7.00000e+00
   3.62000e-01 -7.35100e+00  0.00000e+00  9.31000e-02  8.21740e+01
   6.34000e-01]
 [ 4.56000e-01  7.58000e-01  6.77000e-01  0.00000e+00  5.00000e+00
   9.54000e-02 -6.59800e+00  1.00000e+00  7.82000e-02  1.24998e+02
   7.13000e-01]
 [ 5.26000e-01  8.93000e-01  6.39000e-01  1.77000e-03  1.00000e+01
   1.21000

Normaliser la matrice de features

In [11]:
# Initialiser le scaler
scaler = MinMaxScaler()

# Normaliser la matrice
normalized_features = scaler.fit_transform(features_matrix)

print("Matrice normalisée :\n", normalized_features)


# Calculer la matrice de similarité cosinus
similarity_matrix = cosine_similarity(normalized_features)

Matrice normalisée :
 [[2.96353659e-01 8.06949807e-01 4.98470948e-01 2.70270270e-05
  1.00000000e-01 0.00000000e+00 1.00000000e+00 0.00000000e+00
  6.26816380e-01 2.86230371e-01 6.35346756e-01]
 [5.21979388e-01 6.33204633e-01 3.02752294e-01 0.00000000e+00
  9.00000000e-01 6.71591672e-03 6.94723618e-01 0.00000000e+00
  1.00000000e+00 3.16822586e-01 1.00000000e+00]
 [7.20663875e-02 7.29729730e-01 1.00000000e+00 1.00000000e+00
  5.00000000e-01 6.44056414e-01 8.57726131e-01 0.00000000e+00
  1.97490092e-01 9.57630304e-01 5.72706935e-01]
 [0.00000000e+00 4.13127413e-01 4.77064220e-01 0.00000000e+00
  0.00000000e+00 8.93216924e-02 6.39447236e-01 1.00000000e+00
  2.33817701e-01 2.05479452e-01 0.00000000e+00]
 [1.83540794e-01 0.00000000e+00 1.10091743e-01 1.68648649e-03
  7.00000000e-01 1.00000000e+00 0.00000000e+00 0.00000000e+00
  1.81307794e-01 0.00000000e+00 6.71140940e-01]
 [8.66154229e-01 4.78764479e-01 1.16207951e-01 0.00000000e+00
  5.00000000e-01 1.04768301e-01 2.36494975e-01 1.0000000